# Inference: Pretrained vs Finetuned KumoTabular

We run inference on the `_score` dataset, which was held out and never seen during finetuning. We compare the finetuned model vs the pretrained model on this set. 

This notebook covers three inference approaches:

1. **Pretrained model** — baseline KumoTabular with no task-specific finetuning.
2. **Finetuned model (UC Model Registry)** *(recommended)* — loaded from Unity Catalog via `mlflow.pytorch.load_model()`. Leverages UC governance, versioning, and lifecycle aliases (`@champion`).
3. **Finetuned model (Volume checkpoint)** — loaded from a `.pt` checkpoint stored in a UC Volume via `torch.load()`. Requires a hardcoded `run_id` to locate the checkpoint file. Included as a simpler alternative to get started before learning about MLflow.

All three approaches use in-context learning (ICL): the model sees the training set as context at inference time, then predicts on the test set.

**Compute:** GPU (A10), environment version: AI v6. A10 is enough as here we don't finetune and don't need to backward pass. 

## Configuration and libraries

In [0]:
%pip install git+https://github.com/NVIDIA/structured-data-models.git --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import gc
import os
import sys
import time

import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss
import torch
import sdm

In [0]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

SCHEMA = "default"
VOLUME_NAME = "finetuned_classification_models"
TABLE_NAME_PREFIX = "finetuning_classification"
TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME_PREFIX}_train"
SCORE_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME_PREFIX}_score"
TARGET_COL = "target"
ID_COL = "ID"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks-finetune"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

NUM_ESTIMATORS = 8

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {volume_path}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

In [0]:
# Load train (for ICL context) and score (for evaluation)
train_pdf = spark.table(TRAIN_TABLE).toPandas().dropna(subset=[TARGET_COL])
score_pdf = spark.table(SCORE_TABLE).toPandas().dropna(subset=[TARGET_COL])

all_feature_cols = [c for c in train_pdf.columns if c not in (ID_COL, TARGET_COL)]
feature_cols = all_feature_cols

X_train = train_pdf[feature_cols].reset_index(drop=True)
y_train = train_pdf[[TARGET_COL]].reset_index(drop=True)
X_test = score_pdf[feature_cols].reset_index(drop=True)
y_test = score_pdf[TARGET_COL].values.astype(float)

# Cast numeric columns to float16 to match model precision
for df in (X_train, X_test):
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].astype(np.float16)
test_sample = score_pdf

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Train (context): {len(X_train):,} rows")
print(f"Test:            {len(X_test):,} rows")
print(f"Features:        {len(feature_cols)}")
print(f"Device:          {device}")

## Inference with Pretrained Model

In [0]:
classifier = sdm.models.KumoTabular(task="classification", device=device, size="small", pretrained=True)
classifier.half()  # Use float16 for faster inference and lower GPU memory

X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train, overrides={TARGET_COL: "categorical"}), device=device)

with mlflow.start_run(run_name="kumo_classification_pretrained"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularClassifier",
        "task": "bnp_paribas_cardif",
        "problem_type": "Classification",
        "finetuned": False,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "size": "small",
        "num_estimators": NUM_ESTIMATORS,
        "dtype": "float16",
    })

    t0 = time.time()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        classifier.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=NUM_ESTIMATORS)
    fit_time = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_pdf = classifier.predict(X_test_tensor).to_pandas()
    predict_time = time.time() - t1

    y_pred = pred_pdf.idxmax(axis=1).astype(float).astype(int).values

    results_pdf = test_sample[[ID_COL, TARGET_COL]].copy().reset_index(drop=True)
    results_pdf["predicted"] = y_pred
    for col in pred_pdf.columns:
        results_pdf[f"prob_{col}"] = pred_pdf[col].astype(float).values

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary")
    col_1 = [c for c in pred_pdf.columns if float(c) == 1.0][0]
    y_prob = pred_pdf[col_1].astype(float).values
    auc = roc_auc_score(y_test, y_prob)
    logloss = log_loss(y_test, y_prob)

    mlflow.log_metrics({"accuracy": accuracy, "f1": f1, "auc": auc, "log_loss": logloss, "fit_time_s": fit_time, "predict_time_s": predict_time})
    mlflow.log_table(results_pdf, "score_predictions.json")

    pretrained_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {pretrained_run_id}")
print(f"Fit time:     {fit_time:.1f}s")
print(f"Predict time: {predict_time:.1f}s")
print(f"Accuracy: {accuracy:.6f}")
print(f"F1:       {f1:.6f}")
print(f"AUC:      {auc:.6f}")
print(f"Log Loss: {logloss:.6f}")
display(results_pdf.head(10))

In [0]:
# Free pretrained model before loading finetuned
_gpu_vars = [
    'classifier', 'ft_classifier', 'model',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'pred_pdf', 'pred_pdf_ft',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleaned: {torch.cuda.memory_allocated()/1e6:.0f} MiB allocated")

## Inference with Finetuned Model loading it via UC Model Registry

In [0]:
# Load finetuned model from UC Model Registry (latest version, but you could load any)
from mlflow import MlflowClient

registered_model_name = f"{CATALOG}.{SCHEMA}.kumo_tabular_classification"
client = MlflowClient()
versions = client.search_model_versions(f"name='{registered_model_name}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
model_uri = f"models:/{registered_model_name}/{latest_version}"

ft_classifier = mlflow.pytorch.load_model(model_uri)
ft_classifier = ft_classifier.to(device).half().eval()

print(f"Model: {registered_model_name}")
print(f"Version: {latest_version}")
print(f"URI:   {model_uri}")

In [0]:
X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train, overrides={TARGET_COL: "categorical"}), device=device)

with mlflow.start_run(run_name="kumo_classification_finetuned_mlflow"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularClassifier",
        "task": "bnp_paribas_cardif",
        "problem_type": "Classification",
        "finetuned": True,
        "model_source": "uc_model_registry",
        "registered_model_name": registered_model_name,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "num_estimators": NUM_ESTIMATORS,
        "dtype": "float16",
    })

    t0 = time.time()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        ft_classifier.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=NUM_ESTIMATORS)
    fit_time_mlflow = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_pdf_mlflow = ft_classifier.predict(X_test_tensor).to_pandas()
    predict_time_mlflow = time.time() - t1

    y_pred_mlflow = pred_pdf_mlflow.idxmax(axis=1).astype(float).astype(int).values

    results_pdf_mlflow = test_sample[[ID_COL, TARGET_COL]].copy().reset_index(drop=True)
    results_pdf_mlflow["predicted"] = y_pred_mlflow
    for col in pred_pdf_mlflow.columns:
        results_pdf_mlflow[f"prob_{col}"] = pred_pdf_mlflow[col].astype(float).values

    accuracy_mlflow = accuracy_score(y_test, y_pred_mlflow)
    f1_mlflow = f1_score(y_test, y_pred_mlflow, average="binary")
    col_1 = [c for c in pred_pdf_mlflow.columns if float(c) == 1.0][0]
    y_prob_mlflow = pred_pdf_mlflow[col_1].astype(float).values
    auc_mlflow = roc_auc_score(y_test, y_prob_mlflow)
    logloss_mlflow = log_loss(y_test, y_prob_mlflow)

    mlflow.log_metrics({"accuracy": accuracy_mlflow, "f1": f1_mlflow, "auc": auc_mlflow, "log_loss": logloss_mlflow, "fit_time_s": fit_time_mlflow, "predict_time_s": predict_time_mlflow})
    mlflow.log_table(results_pdf_mlflow, "score_predictions.json")

    mlflow_finetuned_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {mlflow_finetuned_run_id}")
print(f"Fit time:     {fit_time_mlflow:.1f}s")
print(f"Predict time: {predict_time_mlflow:.1f}s")
print(f"Accuracy: {accuracy_mlflow:.6f}")
print(f"F1:       {f1_mlflow:.6f}")
print(f"AUC:      {auc_mlflow:.6f}")
print(f"Log Loss: {logloss_mlflow:.6f}")
display(results_pdf_mlflow.head(10))

In [0]:
# Free MLflow finetuned model before loading checkpoint model
_gpu_vars = [
    'ft_classifier', 'ft_classifier_ckpt',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'pred_pdf_mlflow', 'pred_pdf_ckpt',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleaned: {torch.cuda.memory_allocated()/1e6:.0f} MiB allocated")

## Inference with Finetuned Model via Volume Checkpoint

In [0]:
# Hardcoded run_id from the finetuning run — the checkpoint filename includes it
run_id = "866ccfd541a549e59643b45dbce4006f"

ft_run = mlflow.get_run(run_id)
ft_run_id = ft_run.info.run_id
FT_MODEL_SIZE = ft_run.data.params["model_size"]
ckpt_path = f"{volume_path}/kumo_tabular_{FT_MODEL_SIZE}_classification_finetuned_{ft_run_id}.pt"

assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

print(f"Finetuning run: {ft_run_id}")
print(f"Model size:     {FT_MODEL_SIZE}")
print(f"Checkpoint:     {ckpt_path}")
print(f"Best val acc:   {ft_run.data.metrics.get('ft_best_val_acc', 'N/A')}")

In [0]:
# Load finetuned model from volume checkpoint
ft_classifier_ckpt = sdm.models.KumoTabular(
    task="classification", device=device, size=FT_MODEL_SIZE, pretrained=False,
)
ft_classifier_ckpt.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
ft_classifier_ckpt.half()
ft_classifier_ckpt.eval()

X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train, overrides={TARGET_COL: "categorical"}), device=device)

with mlflow.start_run(run_name="kumo_classification_finetuned_ckpt"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularClassifier",
        "task": "bnp_paribas_cardif",
        "problem_type": "Classification",
        "finetuned": True,
        "model_source": "volume_checkpoint",
        "finetune_checkpoint": ckpt_path,
        "finetune_run_id": run_id,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "size": FT_MODEL_SIZE,
        "num_estimators": NUM_ESTIMATORS,
        "dtype": "float16",
    })

    t0 = time.time()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        ft_classifier_ckpt.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=NUM_ESTIMATORS)
    fit_time_ckpt = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_pdf_ckpt = ft_classifier_ckpt.predict(X_test_tensor).to_pandas()
    predict_time_ckpt = time.time() - t1

    y_pred_ckpt = pred_pdf_ckpt.idxmax(axis=1).astype(float).astype(int).values

    results_pdf_ckpt = test_sample[[ID_COL, TARGET_COL]].copy().reset_index(drop=True)
    results_pdf_ckpt["predicted"] = y_pred_ckpt
    for col in pred_pdf_ckpt.columns:
        results_pdf_ckpt[f"prob_{col}"] = pred_pdf_ckpt[col].astype(float).values

    accuracy_ckpt = accuracy_score(y_test, y_pred_ckpt)
    f1_ckpt = f1_score(y_test, y_pred_ckpt, average="binary")
    col_1 = [c for c in pred_pdf_ckpt.columns if float(c) == 1.0][0]
    y_prob_ckpt = pred_pdf_ckpt[col_1].astype(float).values
    auc_ckpt = roc_auc_score(y_test, y_prob_ckpt)
    logloss_ckpt = log_loss(y_test, y_prob_ckpt)

    mlflow.log_metrics({"accuracy": accuracy_ckpt, "f1": f1_ckpt, "auc": auc_ckpt, "log_loss": logloss_ckpt, "fit_time_s": fit_time_ckpt, "predict_time_s": predict_time_ckpt})
    mlflow.log_table(results_pdf_ckpt, "score_predictions.json")

    ckpt_finetuned_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {ckpt_finetuned_run_id}")
print(f"Fit time:     {fit_time_ckpt:.1f}s")
print(f"Predict time: {predict_time_ckpt:.1f}s")
print(f"Accuracy: {accuracy_ckpt:.6f}")
print(f"F1:       {f1_ckpt:.6f}")
print(f"AUC:      {auc_ckpt:.6f}")
print(f"Log Loss: {logloss_ckpt:.6f}")
display(results_pdf_ckpt.head(10))